# AGender - Step F: figures and tables of the paper (CPU)

Draws every figure and table of the paper from the saved outputs of Steps 2B, 3 and 4. No model is trained
and no number is typed in.

- **Figures** (`figures/`, each as PDF, PNG and the CSV of the plotted values): attackers, conformal sets,
  privacy-utility trade-off, information of each attribute, solver check, and the two churn tables (Step 4).
- **Tables** (`tables/`, LaTeX and CSV): attackers, conformal sets, defences at their operating points,
  paired comparison of the privacy-aware QUBO with every other defence, released attributes; the same for
  Iranian Churn and the audit of IBM Telco (`table_churn_*`).
- **numbers.json**: every number the text quotes, with its 95 % interval over the five repeats.
- **manifest.json**: the input files with their SHA-256, and what was written.

**Kaggle settings:** Accelerator = **None** (CPU). Internet = Off is fine.
**Inputs to add:** `agender-code`, `agender-step2b`, `agender-step3` and `agender-step4` (the output of the
Step 4 notebook). Without `agender-step4` the churn part is skipped and `manifest.json` says so.
**Expected run time:** under a minute.

In [ ]:
import os
import sys
from pathlib import Path

# The AGender code is attached as a Kaggle dataset. Find the `agender` package
# wherever Kaggle placed it, and make it importable.
def find_file(name, root='/kaggle/input'):
    for folder, _, files in os.walk(root, followlinks=True):
        if name in files:
            return Path(folder) / name
    return None


package = find_file('audit.py')
if package is None:
    attached = sorted(str(p) for p in Path('/kaggle/input').glob('*/*'))[:20]
    raise RuntimeError('The agender-code dataset is not attached to this notebook. In the editor: '
                       'Add Input -> Your Datasets -> agender-code, then Save & Run All again. '
                       f'Attached now: {attached or "nothing"}')
if package.parent.name == 'agender' and (package.parent / '__init__.py').exists():
    code_root = package.parent.parent
else:
    # Kaggle sometimes flattens an uploaded folder; link the files under the name `agender`.
    code_root = Path('/kaggle/working/code')
    code_root.mkdir(parents=True, exist_ok=True)
    if not (code_root / 'agender').exists():
        os.symlink(package.parent, code_root / 'agender')

sys.path.insert(0, str(code_root))
import agender
print('AGender code loaded from', Path(agender.__file__).parent)

In [ ]:
# ---- Settings (the only cell you should need to edit) ----
INPUT = Path('/kaggle/input')
OUTPUT = Path('/kaggle/working/agender_outputs/stepF')
TIME_BUDGET_MINUTES = 20

In [ ]:
import json

from agender import figures
from agender.runner import StepRunner

runner = StepRunner(OUTPUT, TIME_BUDGET_MINUTES)
summary = {}
summary['figures'] = runner.run('figures and tables', figures.run, INPUT, OUTPUT)
if summary['figures']:
    print(json.dumps(summary['figures'], indent=1))

In [ ]:
# A quick look at the main figure
from IPython.display import Image, display

main = OUTPUT / 'figures' / 'fig_tradeoff.png'
if main.exists():
    display(Image(filename=str(main)))

In [ ]:
summary['steps'] = runner.steps
with open(OUTPUT / 'stepF_summary.json', 'w') as f:
    json.dump(summary, f, indent=1, default=str)
print('Failed or skipped steps:', runner.failed() or 'none')